In [ ]:
import json
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# ---------------------------------------------------------------- parameters
START_DATE = "2026-06-30"
END_DATE = "2026-07-28"

# Penalty cap rates, in basis points of the volume that failed to settle.
UNCORRELATED_BPS_BY_CHAIN = {
    "ethereum": 4.0, "gnosis": 3.0, "arbitrum": 1.0, "base": 2.0,
    "avalanche_c": 2.0, "polygon": 3.0, "bnb": 1.0,
}
CORRELATED_BPS = 0.1

# Ceiling on a SINGLE failed order's cap. An auction with several failed orders
# can exceed it; one huge order cannot pull the auction's cap up without limit.
PER_ORDER_CAP_USD = 20.0

# CoinGecko spot, 2026-08-06. Fixed rather than a live feed, to keep the
# counterfactual reproducible. Arbitrum and base pay gas in ETH.
NATIVE_TOKEN_USD_PRICE = {
    "ethereum": 1904.08, "arbitrum": 1904.08, "base": 1904.08,
    "gnosis": 1.00, "polygon": 0.075567, "bnb": 592.26, "avalanche_c": 6.42,
}
PER_ORDER_CAP_NATIVE = {
    chain: PER_ORDER_CAP_USD / price
    for chain, price in NATIVE_TOKEN_USD_PRICE.items()
}

# ----------------------------------------------------------------- constants
WEI = 1e18
DATA_DIR = Path("../data")
FETCH_SCRIPT = Path("../scripts/fetch_counterfactual_data.py")
CORRELATED_TOKENS_PATH = DATA_DIR / "correlated_tokens.json"
CORRELATED_TOKENS_URL = "https://cms.cow.finance/api/correlated-tokens?pagination[pageSize]=100"

# Chain -> substrings identifying its groups in the CoW correlated-token lists.
CHAIN_ALIASES = {
    "ethereum": ("mainnet",), "gnosis": ("gnosis",), "arbitrum": ("arbitrum",),
    "base": ("base",), "polygon": ("polygon",), "bnb": ("bnb",),
    "avalanche_c": ("avalanche",),
}

AUCTION_KEYS = ["blockchain", "auction_id", "solver"]
WEEK_KEYS = ["blockchain", "accounting_period"]
SOLVER_WEEK_KEYS = WEEK_KEYS + ["solver"]
KINDS = ("rewards", "failed_orders", "consistency_shares")


def load_inputs(chains):
    """Fetch the three CSVs per chain if missing, and concatenate across chains."""
    frames = {kind: [] for kind in KINDS}
    for chain in chains:
        stem = f"counterfactual_{chain}_{START_DATE}_{END_DATE}"
        paths = {kind: DATA_DIR / f"{stem}_{kind}.csv" for kind in KINDS}
        if not all(path.exists() for path in paths.values()):
            print(f"{chain}: fetching")
            subprocess.run(
                ["uv", "run", "python", str(FETCH_SCRIPT), "--chain", chain,
                 "--start", START_DATE, "--end", END_DATE],
                check=True,
            )
        for kind, path in paths.items():
            frames[kind].append(pd.read_csv(path, low_memory=False))

    return tuple(pd.concat(frames[kind], ignore_index=True) for kind in KINDS)


def load_correlated_groups(chains):
    """Chain -> list of token groups whose members count as correlated pairs."""
    if not CORRELATED_TOKENS_PATH.exists():
        with urllib.request.urlopen(CORRELATED_TOKENS_URL) as response:
            CORRELATED_TOKENS_PATH.write_bytes(response.read())
    published = json.loads(CORRELATED_TOKENS_PATH.read_text())["data"]

    groups = {}
    for chain in chains:
        groups[chain] = [
            {str(token).lower() for token in entry["attributes"]["tokens"]}
            for entry in published
            if any(a in entry["attributes"]["name"].lower() for a in CHAIN_ALIASES[chain])
        ]
        if not groups[chain]:
            raise ValueError(f"No correlated-token groups found for {chain}")
    return groups


def proposed_penalty_caps(failed_orders, groups):
    """Proposed cap per (auction, solver), summed over its failed orders.

    Each order contributes min(rate x its own volume, ceiling).
    """
    failed = failed_orders.copy()
    failed["failed_volume_native"] = failed["failed_volume_native"].astype(float) / WEI

    correlated = np.zeros(len(failed), dtype=bool)
    for chain, token_groups in groups.items():
        on_chain = failed["blockchain"].eq(chain).to_numpy()
        for tokens in token_groups:
            correlated |= (
                on_chain
                & failed["sell_token"].isin(tokens).to_numpy()
                & failed["buy_token"].isin(tokens).to_numpy()
            )

    rate_bps = np.where(
        correlated, CORRELATED_BPS, failed["blockchain"].map(UNCORRELATED_BPS_BY_CHAIN)
    )
    failed["cap"] = np.minimum(
        rate_bps / 1e4 * failed["failed_volume_native"],
        failed["blockchain"].map(PER_ORDER_CAP_NATIVE),
    )
    return failed.groupby(AUCTION_KEYS, as_index=False).agg(
        proposed_cap_native=("cap", "sum")
    )


def build_auctions(rewards, failed_orders, groups):
    """One row per (auction, solver), with each scenario as a column.

    current_native   signed reward under today's flat cap, as actually paid
    proposed_native  signed reward under the proposed cap
    Negative means the solver paid a penalty. Only the penalty side is
    re-capped, so the positive part is one column rather than two.
    """
    auctions = rewards.rename(
        columns={
            "reward_penalty_native": "current_native",
            "reward_penalty_uncapped_native": "uncapped_native",
            "reward_cap_upper_native": "upper_reward_cap_native",
            "is_excluded_from_penalties": "excluded",
        }
    )
    # astype rather than to_numeric: wei values overflow int64 and would stay
    # object dtype, and a bad value should raise rather than become NaN.
    for column in ["current_native", "uncapped_native", "upper_reward_cap_native"]:
        auctions[column] = auctions[column].astype(float) / WEI

    caps = proposed_penalty_caps(failed_orders, groups)
    auctions = auctions.merge(caps, on=AUCTION_KEYS, how="left", validate="one_to_one")
    auctions["proposed_cap_native"] = auctions["proposed_cap_native"].fillna(0.0)

    auctions["positive_reward_native"] = auctions["current_native"].clip(lower=0)
    auctions["current_penalty_native"] = (-auctions["current_native"]).clip(lower=0)
    auctions["proposed_penalty_native"] = np.where(
        auctions["excluded"].fillna(False).astype(bool),
        0.0,
        np.minimum(
            (-auctions["uncapped_native"]).clip(lower=0), auctions["proposed_cap_native"]
        ),
    )
    auctions["proposed_native"] = (
        auctions["positive_reward_native"] - auctions["proposed_penalty_native"]
    )

    # Whatever the upper cap does not pay out funds the consistency budget, so a
    # harsher penalty enlarges the pool that is shared out again.
    for scenario in ["current", "proposed"]:
        auctions[f"{scenario}_budget_native"] = (
            auctions["upper_reward_cap_native"] - auctions[f"{scenario}_native"]
        )
    return auctions


def calculate_solver_payments(auctions, shares):
    """One row per (blockchain, solver), both scenarios side by side."""
    weekly_budget = auctions.groupby(WEEK_KEYS, as_index=False)[
        ["current_budget_native", "proposed_budget_native"]
    ].sum()

    share_sums = shares.groupby(WEEK_KEYS)["consistency_reward_share"].sum()
    assert np.allclose(share_sums, 1.0), share_sums

    allocated = shares.merge(weekly_budget, on=WEEK_KEYS, how="inner")
    for scenario in ["current", "proposed"]:
        allocated[f"{scenario}_consistency_native"] = (
            allocated["consistency_reward_share"]
            * allocated[f"{scenario}_budget_native"]
        )

    weekly_batch = auctions.groupby(SOLVER_WEEK_KEYS, as_index=False).agg(
        positive_reward_native=("positive_reward_native", "sum"),
        current_penalty_native=("current_penalty_native", "sum"),
        proposed_penalty_native=("proposed_penalty_native", "sum"),
        current_batch_native=("current_native", "sum"),
        proposed_batch_native=("proposed_native", "sum"),
    )

    # outer: a solver can hold a consistency v1 share in a week it won no auctions in
    payments = weekly_batch.merge(
        allocated[SOLVER_WEEK_KEYS
                  + ["current_consistency_native", "proposed_consistency_native"]],
        on=SOLVER_WEEK_KEYS,
        how="outer",
    ).fillna(0.0)

    for scenario in ["current", "proposed"]:
        payments[f"{scenario}_total_native"] = (
            payments[f"{scenario}_batch_native"]
            + payments[f"{scenario}_consistency_native"]
        )

    solvers = payments.drop(columns="accounting_period").groupby(
        ["blockchain", "solver"], as_index=False
    ).sum()
    solvers["change_native"] = (
        solvers["proposed_total_native"] - solvers["current_total_native"]
    )
    return solvers


def show_results(solvers):
    """Per-chain table and chart. Solvers are labelled by current-payment rank."""
    for chain in sorted(solvers["blockchain"].unique()):
        table = (
            solvers[solvers["blockchain"].eq(chain)]
            .sort_values("current_total_native", ascending=False)
            .assign(solver=lambda f: [f"solver {n}" for n in range(1, len(f) + 1)])
        )
        print(
            f"\n=== {chain} | uncorrelated={UNCORRELATED_BPS_BY_CHAIN[chain]:g} bps, "
            f"correlated={CORRELATED_BPS:g} bps, per-order cap=${PER_ORDER_CAP_USD:g}"
            f" = {PER_ORDER_CAP_NATIVE[chain]:.6g} native ==="
        )
        display(
            table[
                ["solver", "positive_reward_native",
                 "current_penalty_native", "proposed_penalty_native",
                 "current_consistency_native", "proposed_consistency_native",
                 "current_total_native", "proposed_total_native", "change_native"]
            ].round(6)
        )

        figure = go.Figure()
        figure.add_bar(x=table["solver"], y=table["current_total_native"], name="Current")
        figure.add_bar(x=table["solver"], y=table["proposed_total_native"], name="Proposed")
        figure.add_hline(y=0, line_width=0.8)
        figure.update_layout(
            title=(
                "Solvers PnL Counterfactual — Proposed Penalty Caps — "
                f"{chain} — {START_DATE} to {END_DATE}"
            ),
            xaxis_title="Solver",
            yaxis_title="Total payment (native token)",
            barmode="group",
            template="plotly_white",
            width=max(1100, 30 * len(table) + 300),
            height=700,
        )
        figure.show()


def run_counterfactual():
    chains = sorted(CHAIN_ALIASES)
    rewards, failed_orders, shares = load_inputs(chains)
    groups = load_correlated_groups(chains)

    auctions = build_auctions(rewards, failed_orders, groups)
    solvers = calculate_solver_payments(auctions, shares)

    # Re-capping only moves payments between solvers: either way the chain pays
    # out its upper reward cap. A self-check on the model, not a result.
    totals = solvers.groupby("blockchain")[
        ["current_total_native", "proposed_total_native"]
    ].sum()
    assert np.allclose(totals["current_total_native"], totals["proposed_total_native"])

    show_results(solvers)
    return solvers, auctions


SOLVER_PAYMENTS, AUCTIONS = run_counterfactual()
